In [1]:
!pip install -q datasets transformers evaluate sentence-transformers openai pandas numpy tqdm python-dotenv  google-generativeai requests tqdm -q sacrebleu rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00


In [2]:
import os
import csv
import json
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Other Packages
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm.auto import tqdm
import os
import csv
import os, json, time, math
from tqdm import tqdm
from openai import OpenAI
import re

In [3]:
# mount Google Drive first
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from google.colab import drive

drive.mount('/content/drive')

output_folder = "/content/drive/MyDrive/context_summarization_eval_samples"
import os
os.makedirs(output_folder, exist_ok=True)

# Dataset repos
repos = {
    "GPT-4.1":          "Lakshan2003/GPT-4.1-customerservice-context-summarization-evaldata",
    "Gemini-2.5-Flash": "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-evaldata",
    "Phi-4-Mini":       "Lakshan2003/Phi-4-mini-customerservice-context-summarization-evaldata",
    "Qwen3-4B":         "Lakshan2003/Qwen3-4B-customerservice-context-summarization-evaldata",
    "Qwen3-8B":         "Lakshan2003/Qwen3-8B-customerservice-context-summarization-evaldata",
    "LLaMA-3.1-8B":     "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-evaldata",
    "LLaMA-3.2-3B":     "Lakshan2003/Llama3.2-instruct-customerservice-context-summarization-evaldata",
}

# HuggingFace repos to push LLM judge datasets
hf_judge_repos = {
    "GPT-4.1":          "Lakshan2003/GPT-4.1-customerservice-context-summarization-llm-judge-data",
    "Gemini-2.5-Flash": "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data",
    "Phi-4-Mini":       "Lakshan2003/Phi-4-mini-customerservice-context-summarization-llm-judge-data",
    "Qwen3-4B":         "Lakshan2003/Qwen3-4B-customerservice-context-summarization-llm-judge-data",
    "Qwen3-8B":         "Lakshan2003/Qwen3-8B-customerservice-context-summarization-llm-judge-data",
    "LLaMA-3.1-8B":     "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data",
    "LLaMA-3.2-3B":     "Lakshan2003/Llama3.2-instruct-customerservice-context-summarization-llm-judge-data",
}

SAMPLE_SIZE = 1000
RANDOM_SEED = 42

# Load first dataset and sample 1000 indices
first_df = load_dataset(list(repos.values())[0], split="train").to_pandas()
rng = np.random.default_rng(RANDOM_SEED)
sampled_indices = sorted(rng.choice(len(first_df), size=SAMPLE_SIZE, replace=False))
print(f"Sampled {SAMPLE_SIZE} indices from {len(first_df)} rows")

# Apply same indices to all datasets, save to Drive and push to HuggingFace
for model_name, repo in repos.items():
    df = load_dataset(repo, split="train").to_pandas()
    df_sampled = df.iloc[sampled_indices].reset_index(drop=True)

    # Add scoring columns
    for col in ["Information-Accuracy", "Structural-Clarity", "Faithfulness"]:
        df_sampled[col] = ""

    # Save to Drive
    safe_name = model_name.replace(" ", "_").replace(".", "")
    save_path = os.path.join(output_folder, f"{safe_name}_1000_samples.csv")
    df_sampled.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"Saved {model_name}: {len(df_sampled)} rows -> {save_path}")

    # Push to HuggingFace
    hf_dataset = Dataset.from_pandas(df_sampled)
    hf_dataset.push_to_hub(hf_judge_repos[model_name], private=False)
    print(f"Pushed {model_name} -> {hf_judge_repos[model_name]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sampled 1000 indices from 10000 rows
Saved GPT-4.1: 1000 rows -> /content/drive/MyDrive/context_summarization_eval_samples/GPT-41_1000_samples.csv


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  97%|#########6| 1.40MB / 1.45MB            

Pushed LLaMA-3.2-3B -> Lakshan2003/Llama3.2-instruct-customerservice-context-summarization-llm-judge-data
